# Data imputation

The data is heavely unbalanced, if the nulls are deleted some classes are not even represented. In order to solve this we've used data imputation


In [10]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [11]:
dataset = pd.read_csv("../datasets/thyroid0387_cleanded.data", sep = ";", index_col="Unnamed: 0")

In [12]:
dataset["diagnosis_group"].value_counts()

diagnosis_group
normal                   6547
hypothyroid               572
general_health            435
replacement_therapy       336
binding_protein           324
misc                      247
hyperthyroid              189
antithyroid_treatment      33
Name: count, dtype: int64

In [13]:
dataset.dropna()["diagnosis_group"].value_counts()

diagnosis_group
normal                 15
hypothyroid             2
general_health          1
replacement_therapy     1
Name: count, dtype: int64

# Imputation of NaNs

In [14]:
dataset.dtypes

age                            int64
on_thyroxine                    bool
query_on_thyroxine              bool
on_antithyroid_medication       bool
sick                            bool
pregnant                        bool
thyroid_surgery                 bool
I131_treatment                  bool
query_hypothyroid               bool
query_hyperthyroid              bool
lithium                         bool
goitre                          bool
tumor                           bool
hypopituitary                   bool
psych                           bool
diagnosis                     object
diagnosis_group               object
sex_M                           bool
TSH_value                    float64
T3_value                     float64
TT4_value                    float64
T4U_value                    float64
FTI_value                    float64
TBG_value                    float64
ref_STMW                        bool
ref_SVHC                        bool
ref_SVHD                        bool
r

In [15]:
x = dataset.drop(columns = ["diagnosis_group"])
y = dataset["diagnosis_group"]

In [16]:
numeric_col = x.select_dtypes(include = ["int64", "float64"]).columns
categoric_col = x.select_dtypes(exclude = ["int64", "float64"]).columns

In [17]:
# SimpleImputer can't handle the Bool type, parsing to Object
x[categoric_col] = x[categoric_col].astype("object")

In [18]:
numerical_imputer = SimpleImputer(strategy = "mean")
categorical_imputer = SimpleImputer(strategy = "most_frequent")

In [19]:
processor = ColumnTransformer(
    transformers = [
        ("num", numerical_imputer, numeric_col),
        ("cat", categorical_imputer, categoric_col)
    ]
)

In [20]:
x_imputed = processor.fit_transform(x)
x_imputed = pd.DataFrame(x_imputed, columns = numeric_col.tolist() + categoric_col.tolist())

## Updating dataset

In [21]:
temp = pd.concat([x_imputed, y], axis = 1)

In [22]:
for col in dataset.columns:
    dataset[col] = temp[col]

In [23]:
dataset.dropna()["diagnosis_group"].value_counts()

diagnosis_group
normal                   6171
hypothyroid               543
general_health            416
replacement_therapy       323
binding_protein           305
misc                      239
hyperthyroid              184
antithyroid_treatment      30
Name: count, dtype: int64

In [24]:
dataset = dataset.dropna()

In [26]:
dataset.to_csv("../datasets/thyroid0387_imputed.data", sep=";")